### The Effect of Model Size on Accuracy with Chain of Thought Reasoning Under Inference Time Compute Scaling

### Experiment 1 on tiny language models

In [1]:
!pip install -q transformers datasets accelerate

In [2]:
!pip install -U bitsandbytes>=0.46.1

In [3]:
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoConfig,
    AutoModelForSeq2SeqLM,
    AutoModelForCausalLM,
    BitsAndBytesConfig
)
import torch
import re
from tqdm import tqdm

#### Test data (Summary)

GSM8K (Grade School Math 8K) is a dataset of 8.5K high quality linguistically diverse grade school math word problems. The dataset was created to support the task of question answering on basic mathematical problems that require multi-step reasoning.

- These problems take between 2 and 8 steps to solve.

- Solutions primarily involve performing a sequence of elementary calculations using basic arithmetic operations (+ − ×÷) to reach the final answer.

- A bright middle school student should be able to solve every problem: from the paper, "Problems require no concepts beyond the level of early Algebra, and the vast majority of problems can be solved without explicitly defining a variable."

- Solutions are provided in natural language, as opposed to pure math expressions. From the paper: "We believe this is the most generally useful data format, and we expect it to shed light on the properties of large language models’ internal monologues""

In [6]:
gsm8k = load_dataset("openai/gsm8k", "main", split="test[:50]")

README.md:   0%|          | 0.00/7.93k [00:00<?, ?B/s]

main/train-00000-of-00001.parquet:   0%|          | 0.00/2.31M [00:00<?, ?B/s]

main/test-00000-of-00001.parquet:   0%|          | 0.00/419k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/7473 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1319 [00:00<?, ? examples/s]

In [7]:
gsm8k[0]["question"]

"Janet’s ducks lay 16 eggs per day. She eats three for breakfast every morning and bakes muffins for her friends every day with four. She sells the remainder at the farmers' market daily for $2 per fresh duck egg. How much in dollars does she make every day at the farmers' market?"

In [8]:
gsm8k[0]["answer"]

'Janet sells 16 - 3 - 4 = <<16-3-4=9>>9 duck eggs a day.\nShe makes 9 * 2 = $<<9*2=18>>18 every day at the farmer’s market.\n#### 18'

#### Few shot prompts

In [9]:
few_shots = """Q: Emily has 3 apples. Her friend gives her 2 more. How many apples does Emily have now?
A: Emily starts with 3 apples. Her friend gives her 2 more. So, 3 + 2 = 5. The answer is 5.

Q: A pen costs 2 dollars. John buys 4 pens. How much does he pay?
A: Each pen costs 2 dollars. John buys 4 pens. So, 2 × 4 = 8. The answer is 8.

Q: Jake read 5 pages on Monday and 7 pages on Tuesday. How many pages did he read in total?
A: Jake read 5 pages on Monday and 7 on Tuesday. So, 5 + 7 = 12. The answer is 12.

Q: A bakery bakes 48 cookies and packs them equally into 6 boxes. They sell 3 boxes. How many cookies are left?
A: First, find cookies per box: 48 ÷ 6 = 8 cookies per box. They sell 3 boxes, so cookies sold = 3 × 8 = 24. Cookies remaining = 48 - 24 = 24. The answer is 24.

Q: Maria earns 15 dollars per hour. She works 6 hours on Saturday and 4 hours on Sunday. How much does she earn in total?
A: Saturday earnings = 15 × 6 = 90 dollars. Sunday earnings = 15 × 4 = 60 dollars. Total earnings = 90 + 60 = 150. The answer is 150.
"""

#### Extracting final answer as a single numerical value

In [10]:
def extract_final_answer(text):
    text = text.replace(",", "")
    matches = re.findall(r"\d+(?:\.\d+)?", text)
    if matches:
        return matches[-1]
    return None

#### Loading specific model - seq2seq or casual LM model

In [11]:
def load(model_id, use_4bit = True):
    device = "cuda" if torch.cuda.is_available() else "cpu"

    tokenizer = AutoTokenizer.from_pretrained(model_id)

    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    config = AutoConfig.from_pretrained(model_id)

    if config.is_encoder_decoder:
        model = AutoModelForSeq2SeqLM.from_pretrained(
            model_id,
            dtype=torch.float16 if device == "cuda" else torch.float32
        ).to(device)

        model_type = "seq2seq"

    else:
        if device == "cuda" and use_4bit:
            bnb_config = BitsAndBytesConfig(
                load_in_4bit=True,
                bnb_4bit_compute_dtype=torch.float16
            )

            model = AutoModelForCausalLM.from_pretrained(
                model_id,
                quantization_config=bnb_config,
                device_map="auto"
            )
        else:
            model = AutoModelForCausalLM.from_pretrained(
                model_id,
                dtype=torch.float16 if device == "cuda" else torch.float32
            ).to(device)

        model_type = "causal"

    model.eval()
    return tokenizer, model, model_type, device

In [12]:
def evaluate(model_id, batch_size=8):
    print(f"Evaluating {model_id}")

    tokenizer, model, model_type, device = load(model_id)

    prompts = []
    answers = []

    for qaPair in tqdm(gsm8k, desc="Preparing prompts"):
        question = qaPair["question"]
        answer = qaPair["answer"].split("####")[-1].strip().replace(",", "")

        prompt = few_shots + f"\n\nQ: {question}\nA:"
        prompts.append(prompt)
        answers.append(answer)

    outputs = []

    for i in tqdm(range(0, len(prompts), batch_size), desc="Generating answers"):
        batch_prompts = prompts[i:i + batch_size]

        inputs = tokenizer(
            batch_prompts,
            return_tensors="pt",
            padding=True,
            truncation=True
        )

        if model_type == "seq2seq":
            inputs = inputs.to(device)

        else:
            inputs = inputs.to(model.device)

        with torch.no_grad():
            generated_ids = model.generate(
                **inputs,
                max_new_tokens=128,
                do_sample=False,
                pad_token_id=tokenizer.pad_token_id
            )

        if model_type == "causal":
            input_length = inputs["input_ids"].shape[1]
            generated_ids = generated_ids[:, input_length:]

        batch_texts = tokenizer.batch_decode(
            generated_ids,
            skip_special_tokens=True
        )

        outputs.extend(batch_texts)

    correct = 0

    for generated_text, answer in tqdm(
        zip(outputs, answers),
        total=len(answers),
        desc="Checking accuracy"
    ):
        pred = extract_final_answer(generated_text)

        if pred == answer:
            correct += 1

    total = len(gsm8k)
    accuracy = correct / total

    print(f"Accuracy: {accuracy}")
    return accuracy

In [13]:
import plotly.express as px
import plotly.io as pio
pio.renderers.default = "notebook_connected"
model_ids = {
    "SmolLM2 135M": "HuggingFaceTB/SmolLM2-135M-Instruct",
    "SmolLM2 360M": "HuggingFaceTB/SmolLM2-360M-Instruct",
    "Flan-T5 Small": "google/flan-t5-small",
    "Flan-T5 Base": "google/flan-t5-base"
}

results = []

for label, model_id in model_ids.items():
    acc = evaluate(model_id, batch_size=8)
    results.append({
        "Model": label,
        "Accuracy": acc
    })

fig = px.bar(
    results,
    x="Model",
    y="Accuracy",
    text="Accuracy",
    title="Few-Shot CoT Reasoning Accuracy vs Model Size (GSM8K)",
)

fig.update_traces(
    texttemplate="%{text}",
    textposition="outside"
)

fig.update_layout(
    yaxis=dict(
        title="Accuracy",
        range=[0, 1]
    ),
    xaxis_title="Model",
    template="plotly_white"
)

fig.show()

Evaluating HuggingFaceTB/SmolLM2-135M-Instruct


config.json:   0%|          | 0.00/861 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/3.76k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.10M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/655 [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Checking accuracy: 100%|██████████| 50/50 [00:00<00:00, 49309.95it/s]


Accuracy: 0.04
